### Imports

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

import opinf

import config
import utils
import step1_generate_data as step1

Logging to log.log


### Setup

In [2]:
training_span = (0,1)
num_samples = 20
noiselevel = .05
num_regression_points = 80
numPODmodes = 5
openonsave = False 

time_domain = config.time_domain

In [3]:
# Step 1: Generate data ---------------------------------------------------
sampler = step1.TrajectorySampler(
    training_span,
    num_samples,
    noiselevel,
    num_regression_points,
    synced=False,
)
(
    true_states,
    time_domain_sampled,
    snapshots_sampled,
    training_inputs,
) = sampler.multisample(config.input_parameters, plot=openonsave)

INFO:2025-06-16 12:29:03,671:jax._src.xla_bridge:867: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/anthonypoole/miniconda3/envs/jax/bin/../lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)


In [4]:
trajectory = 0
true_state = true_states[0]
time_domain_sampled = time_domain_sampled[0]
snapshots_sampled = snapshots_sampled[0]
training_inputs = training_inputs[0]
print(true_state.shape, time_domain_sampled.shape, snapshots_sampled.shape, training_inputs.shape)

(500, 500) (20,) (500, 20) (2, 80)


In [5]:
with opinf.utils.TimedBlock(
    f"reducing noisy training states to {numPODmodes} dimensions"
    ):
    basis = config.Basis(num_vectors=numPODmodes)
    basis.fit(snapshots_sampled)
    snapshots_compressed = basis.compress(snapshots_sampled)
    print(snapshots_compressed.shape)

reducing noisy training states to 5 dimensions...shift thing (1000, 20) (1000, 1)
(1000, 20) (1000,)
shift thing (1000, 20) (1000, 1)
(5, 20)
done in 0.00 s.


In [6]:
import jax.numpy as jnp
input_func = config.input_func_factory(config.input_parameters[0])
time_domain_sampled = time_domain_sampled
inputs = input_func(time_domain_sampled) 

rom = opinf.ROM(
    basis=basis,
    ddt_estimator=opinf.ddt.NonuniformFiniteDifferencer(time_domain_sampled),
    model=opinf.models.ContinuousModel(
        operators="cAHBN",
        solver=opinf.lstsq.L2Solver(regularizer=1e-6),
    )
).fit(states=snapshots_sampled, inputs=inputs)

print(true_state[:,0].shape)
Q_rom_single = rom.predict(jnp.array(true_state[:, 0][:,None]), jnp.array(time_domain), input_func=input_func)

shift thing (1000, 20) (1000, 1)
(1000, 20) (1000,)
shift thing (1000, 20) (1000, 1)


/Users/anthonypoole/Repositories/rom-operator-inference-Python3/src/opinf/lstsq/_tikhonov.py:79: OpInfWarning: non-regularized least-squares system is underdetermined
  warnings.warn(


(500,)
(1000, 1) (1000,)
shift thing (1000, 1) (1000, 1)


ValueError: Terms are not compatible with solver! Got:
ODETerm(vector_field=<function ContinuousModel.predict.<locals>.<lambda>>)
but expected:
diffrax.AbstractTerm
Note that terms are checked recursively: if you scroll up you may find a root-cause error that is more specific.

In [ ]:
# # Step 2: Fit Gaussian processes to data ----------------------------------
# # Dimensionality reduction with POD.
# with opinf.utils.TimedBlock(
#     f"reducing noisy training states to {numPODmodes} dimensions"
# ):
#     basis = config.Basis(num_vectors=numPODmodes)
#     basis.fit(np.hstack(snapshots_sampled))
#     # # Some plotting stuff
#     # ax = basis.plot_svdval_decay()
#     # ax.set_xlim(right=20)
#     # ax.set_ylim(bottom=1e-4)
#     # utils.save_figure("svdvals.pdf", andopen=openonsave)
#     snapshots_compressed = [basis.compress(Q) for Q in snapshots_sampled]
#     print(np.array(snapshots_compressed).shape, np.array(snapshots_sampled).shape)

In [ ]:
# plt.figure(figsize=(12,8))
# plt.imshow(np.array(snapshots_sampled)[:,:,0])
# plt.show()
# plt.imshow(np.array(snapshots_compressed)[:,:,0])
# plt.show()

In [ ]:
# import random
# import time

# ### Generate Prior For operator matrix
# a,b = config.input_parameters[0]
# input_func = config.input_func_factory(config.input_parameters[0])
# time_domain_sampled = np.array(time_domain_sampled)
# inputs = input_func(time_domain_sampled[0,:]) 

# basis = config.Basis(num_vectors=numPODmodes)

# rom = opinf.ROM(
#     basis=basis,
#     ddt_estimator=opinf.ddt.NonuniformFiniteDifferencer(time_domain_sampled[0,:]),
#     model=opinf.models.ContinuousModel(
#         operators="cAHBN",
#         solver=opinf.lstsq.L2Solver(regularizer=1e-6),
#     )
# ).fit(states=snapshots_sampled, inputs=inputs)

# print(rom.model.operator_matrix.shape)

# # Solve the ROM over a specified time domain. Make sure it's stable!
# true_states = np.array(true_states)
# print(f'State size', true_states.shape)
# print(f'Initial state size', true_states[:,:,0].shape)
# Q_rom_list = []
# Q_rom_single = rom.predict(true_states[:, :, 0], time_domain, input_func=input_func)

# prior_ohat = rom.model.operator_matrix

# # Put the operator in the ROM
# rom.model._extract_operators(np.array(prior_ohat))

# # Plot predictions within training time domains
# deter_pred = rom.model.predict(
#     state0=snapshots_compressed[:, 0], 
#     t=time_domain_sampled, 
#     input_func=input_func
# )
# deter_sol = rom.model.predict_result_
# print(deter_sol.ts.shape)          
# # print(deter_sol.message) 

# if deter_sol.ts.shape[0] == snapshots_sampled.shape[1]:
#     print("me dont know")
# else:
#     print(deter_sol.t.shape[0], snapshots_compressed.shape, sep='\n')
#     print("Failed to find stable operator, retrying...")
#     del rom